# 3D Game of Life (CUDA)

Conway's Game of Life in a cubic voxel box, on the GPU via `numba.cuda`, with the same kind of viewer as the nucleation notebook. Each step, every voxel counts the live cells around it, and all voxels update at once:
* **Creation (birth):** a dead voxel becomes a live cell if the count is in the birth range.
* **Sustenance (survival):** a live cell stays alive (and ages by one step) if the count is in the survival range.
* **Death:** otherwise a live cell dies: of isolation when the count is below the survival range, of overcrowding when it is above.

The four numbers (birth min/max, survival min/max) are set in the viewer, with a few presets per window. Each window keeps its own rule. The rule shows in the usual B/S notation, e.g. B5-6/S4-6.

**Window (toggle):**
* **3x3x3** (default): the 26 voxels around the cell (counts 0 to 26), the 3D version of Conway's 8-neighbour square. Default B5-6/S4-6, which stays busy; Carter Bays' classic B5/S4-5 ("4555", known for its gliders) is a preset, but random soups mostly freeze under it.
* **2x2x2**: the box is cut into 2x2x2 blocks, and the count is the live cells among the other 7 in the cell's block (0 to 7). The blocks shift by one voxel along every axis each step (as in a Margolus neighbourhood), so patterns can spread across block borders. Default B3-4/S2-4. With wrap on, use an even box side for an exact block partition.

**Wrap edges (toggle, on by default):** opposite faces of the box meet (a torus); off, everything outside the box counts as dead.

**Size (viewer, applied with a reset):** the box side (4 to 512 voxels, default 96), the number of starting cells (default 20,000), and the side of the central cube they are scattered in at random, as a fraction of the box (default 40%; the cell count is capped at that cube's volume).

View: cells are drawn as voxels coloured by age, from newborn yellow to old dark red, and shaded by depth. The opacity slider makes them translucent to see inside a blob. **Drag** to rotate, **scroll** to zoom, **double-click** to reset the view. **Step** advances one step while paused. The info line shows the live cells and how many were created and died in the last step.

How it works on the GPU (MX330: ~370 steps/s at 128³, ~40 at 256³, ~5-9 at 512³):
* The box is a dense byte grid (0 = dead, else the cell's age, up to 255), double-buffered.
* The 3x3x3 count is separable: sums along x, then y, then z, 9 reads per voxel instead of 27. The rule is applied in the z pass.
* Kernels use a 3D thread grid, so each thread gets its x, y, z without integer division.
* Births, deaths and the list of live cells for the viewer are gathered a warp at a time (one atomic per warp). The viewer gets 4 bytes per live cell (position and age class), up to 3 million cells per frame, and only once it has drawn the last frame.
* The browser draws with a depth buffer instead of sorting.

Run the cells in order. No extra packages beyond `numba` and `numpy`.


In [ ]:
import json, threading, time, warnings
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse
from IPython.display import HTML, IFrame, display
from numba import cuda

warnings.filterwarnings('ignore', category=Warning, module='numba')
print(cuda.get_current_device().name.decode())

In [ ]:
import math
import numpy as np
from numba import cuda, int32, uint32

LMAX = 512                # largest box side (GPU memory: 4 bytes per voxel, 0.5 GB at 512)
NMAX = 50_000_000         # most starting cells
SHOW = 3_000_000          # most live cells sent to the viewer per frame
PB = 9                    # bits per coordinate in a packed cell: x | y<<PB | z<<2PB | age class<<27
TB = (32, 4, 2)            # thread block: one warp along x
LAUNCH = 0.05             # seconds of GPU work between checks of the time budget (steps per check adapt)
RULES = {3: (5, 6, 4, 6),  # default (birth min, birth max, survive min, survive max) for each window
         2: (3, 4, 2, 4)}
assert LMAX <= 1 << PB


@cuda.jit(device=True)
def apply(age, c, bmask, smask):
    """New state of a cell of this age (0 = dead) with c live cells around it: a live cell survives (and
    ages, up to 255) if c is in the survive set, else dies; a dead cell is born if c is in the birth set."""
    if age:
        if (smask >> c) & 1:
            return age + 1 if age < 255 else 255
        return 0
    return 1 if (bmask >> c) & 1 else 0


@cuda.jit(device=True)
def tally(stats, par, born, died):
    """Births and deaths of this step, counted per warp first (one atomic per warp)."""
    mb = cuda.ballot_sync(0xFFFFFFFF, born)
    md = cuda.ballot_sync(0xFFFFFFFF, died)
    if cuda.laneid == 0:
        if mb:
            cuda.atomic.add(stats, 2 * par, cuda.popc(mb))
        if md:
            cuda.atomic.add(stats, 2 * par + 1, cuda.popc(md))


@cuda.jit
def sum_axis(src, dst, L, axis, wrap, first):
    """dst = src summed over each voxel and its two neighbours along one axis (0 = x, 1 = y, 2 = z): one
    pass of the separable 3x3x3 count. The first pass sums live flags (src = ages), later ones partial sums.
    3D launch, so x, y, z come straight from the thread index (no slow integer division)."""
    x, y, z = cuda.grid(3)
    if x >= L or y >= L or z >= L:
        return
    i = (z * L + y) * L + x
    c = x if axis == 0 else (y if axis == 1 else z)
    stride = 1 if axis == 0 else (L if axis == 1 else L * L)
    v = src[i]
    s = (1 if v else 0) if first else v
    j = i - stride if c > 0 else (i + (L - 1) * stride if wrap else -1)
    if j >= 0:
        v = src[j]
        s += (1 if v else 0) if first else v
    j = i + stride if c < L - 1 else (i - (L - 1) * stride if wrap else -1)
    if j >= 0:
        v = src[j]
        s += (1 if v else 0) if first else v
    dst[i] = s


@cuda.jit
def rule3(g, b, g2, L, wrap, bmask, smask, stats, par):
    """3x3x3 window: the last (z) pass of the count, then the rule. c = live cells among the 26 around."""
    x, y, z = cuda.grid(3)
    LL = L * L
    born = False
    died = False
    if x == 0 and y == 0 and z == 0:
        stats[2 - 2 * par] = 0                                         # the other step parity's tally
        stats[3 - 2 * par] = 0
    if x < L and y < L and z < L:
        i = (z * L + y) * L + x
        s = b[i]
        j = i - LL if z > 0 else (i + (L - 1) * LL if wrap else -1)
        if j >= 0:
            s += b[j]
        j = i + LL if z < L - 1 else (i - (L - 1) * LL if wrap else -1)
        if j >= 0:
            s += b[j]
        age = g[i]
        new = apply(age, s - (1 if age else 0), bmask, smask)
        g2[i] = new
        born = age == 0 and new > 0
        died = age > 0 and new == 0
    tally(stats, par, born, died)                                      # (every lane of every warp gets here)


@cuda.jit
def rule2(g, g2, L, wrap, o, bmask, smask, stats, par):
    """2x2x2 window (Margolus-style blocks): the box is cut into 2x2x2 blocks, shifted by one voxel on every
    axis each step (o = step parity), and c = live cells among the other 7 in the cell's block."""
    x, y, z = cuda.grid(3)
    born = False
    died = False
    if x == 0 and y == 0 and z == 0:
        stats[2 - 2 * par] = 0
        stats[3 - 2 * par] = 0
    if x < L and y < L and z < L:
        x0 = x - ((x + o) & 1)                                         # the block's lowest corner
        y0 = y - ((y + o) & 1)
        z0 = z - ((z + o) & 1)
        s = 0
        for dz in range(2):
            zz = z0 + dz
            if zz < 0 or zz >= L:
                if not wrap:
                    continue
                zz = L - 1 if zz < 0 else 0
            for dy in range(2):
                yy = y0 + dy
                if yy < 0 or yy >= L:
                    if not wrap:
                        continue
                    yy = L - 1 if yy < 0 else 0
                for dx in range(2):
                    xx = x0 + dx
                    if xx < 0 or xx >= L:
                        if not wrap:
                            continue
                        xx = L - 1 if xx < 0 else 0
                    if g[(zz * L + yy) * L + xx]:
                        s += 1
        i = (z * L + y) * L + x
        age = g[i]
        new = apply(age, s - (1 if age else 0), bmask, smask)
        g2[i] = new
        born = age == 0 and new > 0
        died = age > 0 and new == 0
    tally(stats, par, born, died)


@cuda.jit
def pack(g, out, cnt, L, cap):
    """Live cells -> out, packed x | y<<PB | z<<2PB | age class<<27 (age 1-16 -> 0-15, then 16 more classes
    of 16 steps), appended in warp-sized batches with one atomic per warp. cnt[0]: all live cells (out
    holds the first `cap`)."""
    x, y, z = cuda.grid(3)
    age = int32(g[(z * L + y) * L + x]) if x < L and y < L and z < L else 0
    on = age != 0
    m = cuda.ballot_sync(0xFFFFFFFF, on)
    lane = cuda.laneid
    base = 0
    if lane == 0 and m:
        base = cuda.atomic.add(cnt, 0, cuda.popc(m))
    base = cuda.shfl_sync(0xFFFFFFFF, base, 0)
    if on:
        q = base + cuda.popc(m & ((1 << lane) - 1))
        if q < cap:
            cls = age - 1 if age <= 16 else 16 + (age - 17) // 16
            out[q] = uint32(x | (y << PB) | (z << (2 * PB)) | (cls << 27))


def sample(V, N, rng):
    """N distinct random indices out of range(V)."""
    if 2 * N > V:                                                      # dense: pick the ones left out
        keep = np.ones(V, bool)
        keep[sample(V, V - N, rng)] = False
        return np.flatnonzero(keep)
    idx = np.unique(rng.integers(0, V, N))
    while len(idx) < N:
        idx = np.unique(np.concatenate([idx, rng.integers(0, V, N - len(idx))]))
    return idx


def need(L):
    """GPU bytes a box of side L takes: two age grids, two partial-sum grids and the viewer's buffer."""
    return 4 * L ** 3 + 4 * min(SHOW, L ** 3) + (1 << 20)


def mask(lo, hi):
    """Bit set of the counts lo..hi (empty if lo > hi)."""
    return sum(1 << c for c in range(max(lo, 0), min(hi, 26) + 1))


class Life:
    window = 3                # neighbourhood: 3 = 3x3x3 (26 around), 2 = 2x2x2 blocks (7 around)
    wrap = True               # opposite faces of the box meet (else everything outside is dead)

    def __init__(self, L=96, N=20_000, soup=0.4, seed=0):
        self.L = L = int(min(max(L, 4), LMAX))
        s = int(min(max(round(soup * L), 1), L))                       # starting cells fill a central cube
        self.N = N = int(min(max(N, 0), NMAX, s ** 3))
        self.rules = {k: list(v) for k, v in RULES.items()}
        rng = np.random.default_rng(seed)
        idx = sample(s ** 3, N, rng)
        o = (L - s) // 2
        x, y, z = idx % s + o, (idx // s) % s + o, idx // (s * s) + o
        g = np.zeros(L ** 3, np.uint8)
        g[(z * L + y) * L + x] = 1
        self.d_g = cuda.to_device(g)
        self.d_g2 = cuda.device_array(L ** 3, np.uint8)
        self.d_a = cuda.device_array(L ** 3, np.uint8)                 # partial sums of the 3x3x3 count
        self.d_b = cuda.device_array(L ** 3, np.uint8)
        self.d_out = cuda.device_array(min(SHOW, L ** 3), np.uint32)
        self.d_cnt = cuda.device_array(1, np.int32)
        self.d_stats = cuda.to_device(np.zeros(4, np.int64))           # [births, deaths] x step parity
        self.blocks = tuple((L + b - 1) // b for b in TB)
        self.masks = {}                                                # rule -> (birth set, survive set)
        self.t = 0
        self.chunk = 1                                                 # steps between time checks (adapts)

    def step(self):
        L, blocks, par = self.L, self.blocks, self.t & 1
        rule = tuple(self.rules[self.window])
        if rule not in self.masks:
            self.masks[rule] = (mask(rule[0], rule[1]), mask(rule[2], rule[3]))
        bm, sm = self.masks[rule]
        if self.window == 3:
            sum_axis[blocks, TB](self.d_g, self.d_a, L, 0, int(self.wrap), 1)         # x
            sum_axis[blocks, TB](self.d_a, self.d_b, L, 1, int(self.wrap), 0)         # y
            rule3[blocks, TB](self.d_g, self.d_b, self.d_g2, L, int(self.wrap), bm, sm, self.d_stats, par)
        else:
            rule2[blocks, TB](self.d_g, self.d_g2, L, int(self.wrap), par, bm, sm, self.d_stats, par)
        self.d_g, self.d_g2 = self.d_g2, self.d_g
        self.t += 1

    def run(self, steps, budget=None):
        """Run up to `steps` steps; with a budget (seconds), stop early once it is used up."""
        t_start = time.perf_counter()
        while steps > 0:
            n = min(steps, self.chunk)
            t0 = time.perf_counter()
            for _ in range(n):
                self.step()
            cuda.synchronize()
            now = time.perf_counter()
            self.chunk = max(1, min(1000, int(n * LAUNCH / max(now - t0, 1e-5))))
            steps -= n
            if budget is not None and now - t_start > budget:
                break

    def frame(self):
        """The live cells for the viewer (4 bytes each), and counts for the info line."""
        self.d_cnt[:] = 0
        pack[self.blocks, TB](self.d_g, self.d_out, self.d_cnt, self.L, len(self.d_out))
        live = int(self.d_cnt.copy_to_host()[0])
        shown = min(live, len(self.d_out))
        st = self.d_stats.copy_to_host()
        p = (self.t - 1) & 1                                           # the last step's tally
        data = self.d_out[:shown].copy_to_host().tobytes() if shown else b''
        info = dict(t=self.t, L=self.L, N=self.N, live=live, shown=shown,
                    births=int(st[2 * p]) if self.t else 0, deaths=int(st[2 * p + 1]) if self.t else 0)
        return data, info


for w in (3, 2):                                                       # compile kernels up front
    sim = Life(L=8, N=50)
    sim.window = w
    sim.run(2)
    sim.frame()
print('kernels compiled')


In [ ]:
# The viewer is a plain web page served from this kernel (no Jupyter widgets, which VS Code fails to load here).
# It polls /frame for the packed live cells and sends control changes to /ctl.
import gc
PAGE = r"""<!doctype html><html><head><meta charset="utf-8"><style>
  body { margin: 0; background: #1b1d22; color: #eee; font: 13px system-ui, sans-serif; }
  #bar { display: flex; gap: 10px; align-items: center; padding: 8px; flex-wrap: wrap; }
  .row { display: flex; gap: 10px; align-items: center; padding: 0 8px 8px; flex-wrap: wrap; }
  button, select { font: inherit; padding: 3px 10px; }
  input[type=number] { font: inherit; width: 64px; }
  .wide { width: 96px !important; }
  .tog { min-width: 84px; }  .tog.on { background: #2f6bff; color: #fff; border: 1px solid #2f6bff; }
  canvas { display: block; cursor: grab; touch-action: none; }
  #info { padding: 0 8px 6px; font-variant-numeric: tabular-nums; }
  .dim { opacity: .7; }
</style></head><body>
<div id="bar">
  steps/frame <input id="speed" type="range" min="0" max="3" step="0.1" value="0"> <span id="sv">1</span>
  <button id="pause">Pause</button> <button id="step">Step</button> <button id="reset">Reset</button> <button id="stop">Stop</button>
  <span class="dim">drag = rotate, scroll = zoom, double-click = reset view</span>
</div>
<div class="row">
  box <input id="Lsz" type="number" min="4" max="512" value="96"> voxels a side,
  <input id="Nsz" class="wide" type="number" min="0" max="50000000" step="1000" value="20000"> starting cells
  in a central cube <input id="soup" type="range" min="1" max="100" step="1" value="40"> <span id="soupv">40%</span> of the box
  <span class="dim">(applied on Reset)</span>
</div>
<div class="row">
  <button id="win" class="tog">Window: 3x3x3</button>
  <button id="wrap" class="tog on">Wrap edges: on</button>
  &nbsp; birth (creation) <input id="b0" type="number" min="0" max="26"> to <input id="b1" type="number" min="0" max="26">
  &nbsp; survival (sustenance) <input id="s0" type="number" min="0" max="26"> to <input id="s1" type="number" min="0" max="26">
  <select id="preset"></select>
</div>
<div class="row"><span id="rule" class="dim"></span></div>
<div class="row">
  opacity <input id="op" type="range" min="5" max="100" step="1" value="100"> <span id="opv">100%</span>
  <span class="dim">colour = age: newborn yellow &rarr; old dark red</span>
</div>
<div id="info">waiting for the simulation...</div>
<canvas id="cv"></canvas>
<script>
// Drawing: every live cell is splatted into a pixel buffer with a depth buffer (no per-frame sort), shaded
// by age and by depth. Below 100% opacity the cells are blended instead, so the inside of a blob shows.
const S = 600, PB = 9, PM = (1 << PB) - 1;
const cv = document.getElementById('cv'), dpr = window.devicePixelRatio || 1, W = Math.round(S * dpr);
cv.width = cv.height = W; cv.style.width = cv.style.height = S + 'px';
const ctx = cv.getContext('2d');
const img = ctx.createImageData(W, W), pix = new Uint32Array(img.data.buffer), zb = new Float32Array(W * W);
const rgb = (r, g, b) => ((255 << 24) | (b << 16) | (g << 8) | r) >>> 0;              // (little-endian RGBA)
const BGC = rgb(0x1b, 0x1d, 0x22);
function hsl(hd, s, l) {
  const f = k => { const a = s * Math.min(l, 1 - l), m = (k + hd / 30) % 12; return 255 * (l - a * Math.max(-1, Math.min(m - 3, 9 - m, 1))); };
  return [f(0), f(8), f(4)];
}
const NSH = 8, LUT = new Uint32Array(32 * NSH);                    // age class x depth shade -> colour
for (let q = 0; q < 32; q++) {
  const t = Math.min(1, q / 30), c = hsl((415 - 75 * t) % 360, 0.95, 0.72 - 0.38 * t);   // yellow -> dark red
  for (let k = 0; k < NSH; k++) { const f = 0.45 + 0.55 * k / (NSH - 1); LUT[q * NSH + k] = rgb(c[0] * f | 0, c[1] * f | 0, c[2] * f | 0); }
}
function blend(c0, c1, a) {
  const b = 1 - a;
  return rgb(((c0 & 255) * b + (c1 & 255) * a) | 0, (((c0 >>> 8) & 255) * b + ((c1 >>> 8) & 255) * a) | 0,
             (((c0 >>> 16) & 255) * b + ((c1 >>> 16) & 255) * a) | 0);
}
let L = 96, cells = new Uint32Array(0), fid = -1, synced = false, az = 0.6, tilt = 0.4, zoom = 1, drag = null, queued = false, paused = false;
let OP = 1;

function draw() {
  queued = false;
  const h = (L - 1) / 2, sc = zoom * W / (L * 1.8), c2 = W / 2, dmax = L * 0.87;
  const ca = Math.cos(az), sa = Math.sin(az), ce = Math.cos(tilt), se = Math.sin(tilt);
  const proj = (x, y, z) => {                                       // z is up; az turns about z
    const dx = x - h, dy = y - h, dz = z - h, x1 = ca * dx - sa * dy, d1 = -(sa * dx + ca * dy);
    return [c2 + x1 * sc, c2 - (ce * dz - se * d1) * sc];
  };
  pix.fill(BGC); zb.fill(-Infinity);
  const sz = Math.max(1, Math.round(Math.max(dpr, sc))), opaque = OP >= 1;
  for (let i = 0, n = cells.length; i < n; i++) {
    const p = cells[i], dx = (p & PM) - h, dy = ((p >>> PB) & PM) - h, dz = ((p >>> (2 * PB)) & PM) - h;
    const x1 = ca * dx - sa * dy, d1 = -(sa * dx + ca * dy), depth = se * dz + ce * d1;   // larger = nearer
    const sx = c2 + x1 * sc, sy = c2 - (ce * dz - se * d1) * sc;
    let k = ((depth / dmax + 1) * 0.5 * NSH) | 0; k = k < 0 ? 0 : (k >= NSH ? NSH - 1 : k);
    const col = LUT[(p >>> 27) * NSH + k];
    const x0 = Math.round(sx - sz / 2), y0 = Math.round(sy - sz / 2);
    const xa = x0 < 0 ? 0 : x0, xb = x0 + sz > W ? W : x0 + sz, ya = y0 < 0 ? 0 : y0, yb = y0 + sz > W ? W : y0 + sz;
    for (let yy = ya; yy < yb; yy++) {
      for (let o = yy * W + xa, oe = yy * W + xb; o < oe; o++) {
        if (opaque) { if (depth > zb[o]) { pix[o] = col; zb[o] = depth; } }
        else pix[o] = blend(pix[o], col, OP);
      }
    }
  }
  ctx.putImageData(img, 0, 0);
  ctx.strokeStyle = 'rgba(255,255,255,0.25)'; ctx.lineWidth = dpr; ctx.beginPath();     // cube outline
  for (const [a, b] of [[0,1],[2,3],[4,5],[6,7],[0,2],[1,3],[4,6],[5,7],[0,4],[1,5],[2,6],[3,7]]) {
    const pa = proj(a & 1 ? L - 1 : 0, a & 2 ? L - 1 : 0, a & 4 ? L - 1 : 0);
    const pb = proj(b & 1 ? L - 1 : 0, b & 2 ? L - 1 : 0, b & 4 ? L - 1 : 0);
    ctx.moveTo(pa[0], pa[1]); ctx.lineTo(pb[0], pb[1]);
  }
  ctx.stroke();
}
const schedule = () => { if (!queued) { queued = true; requestAnimationFrame(draw); } };

// Rules: one (birth min, birth max, survive min, survive max) per window; c counts the live cells around a
// cell (26 for 3x3x3, the 7 others in its block for 2x2x2).
const PRESETS = {
  3: [['B5-6/S4-6: busy (default)', [5, 6, 4, 6]], ['B5/S4-5: Bays 4555, gliders, soups freeze', [5, 5, 4, 5]],
      ['B6/S5-7: Bays 5766', [6, 6, 5, 7]], ['B4/S4-5: grows', [4, 4, 4, 5]], ['B5-7/S6-8: dense churn', [5, 7, 6, 8]]],
  2: [['B3-4/S2-4: busy (default)', [3, 4, 2, 4]], ['B3/S2-3: Conway-like, dies out', [3, 3, 2, 3]],
      ['B3-4/S3-5: sparse churn', [3, 4, 3, 5]], ['B2-3/S2-4: dense', [2, 3, 2, 4]], ['B3/S1-3: freezes', [3, 3, 1, 3]]],
};
let rules = { 3: [5, 6, 4, 6], 2: [3, 4, 2, 4] }, win = 3;
const rng = (a, b) => a > b ? 'none' : a === b ? `${a}` : `${a}-${b}`;
function showRule() {
  const [b0, b1, s0, s1] = rules[win], nmax = win === 3 ? 26 : 7;
  ['b0', 'b1', 's0', 's1'].forEach((id, k) => { const el = document.getElementById(id); el.max = nmax; el.value = rules[win][k]; });
  const pre = document.getElementById('preset');
  pre.innerHTML = '<option value="">presets...</option>' + PRESETS[win].map(([name], k) => `<option value="${k}">${name}</option>`).join('');
  document.getElementById('rule').textContent =
    `B${rng(b0, b1)}/S${rng(s0, s1)}: counting the live cells around a cell (0 to ${nmax}), a dead cell is created if the count is ${rng(b0, b1)}, ` +
    `a live cell is sustained if it is ${rng(s0, s1)}, and dies of isolation below ${s0} or of overcrowding above ${s1}.`;
}
function sendRule() { ctl(`window=${win}&rule=${rules[win].join(',')}`); showRule(); }

async function poll() {
  try {
    const r = await fetch('frame?have=' + fid, { cache: 'no-store' });
    const st = JSON.parse(r.headers.get('X-State')), body = await r.arrayBuffer();
    if (st.t === undefined) { setTimeout(poll, 100); return; }                          // no frame yet
    if (!synced) {                                                  // show the simulation's real settings
      synced = true;
      rules = st.rules; win = st.window;
      setTog(winB, win === 3, 'Window', win === 3 ? '3x3x3' : '2x2x2'); setTog(wrapB, st.wrap, 'Wrap edges');
      showRule();
      Lsz.value = st.L; Nsz.value = st.N;
      soupS.value = Math.round(100 * st.soup); document.getElementById('soupv').textContent = soupS.value + '%';
      speed.value = Math.log10(st.speed); document.getElementById('sv').textContent = st.speed;
    }
    if (body.byteLength) { L = st.L; cells = new Uint32Array(body); fid = st.fid; schedule(); }   // redraw only on new data
    document.getElementById('info').textContent = st.error ? 'simulation stopped: ' + st.error :
      `box ${st.L}³   step ${st.t.toLocaleString()}   live cells ${st.live.toLocaleString()}` +
      (st.shown < st.live ? ` (showing ${st.shown.toLocaleString()})` : '') +
      `   last step: ${st.births.toLocaleString()} created, ${st.deaths.toLocaleString()} died` + (st.alive ? '' : '   [stopped]') +
      (st.notice ? '   (' + st.notice + ')' : '');
  } catch (e) { document.getElementById('info').textContent = 'lost connection to the kernel (rerun the cell)'; }
  setTimeout(poll, 33);
}
const ctl = q => fetch('ctl?' + q);
const setTog = (b, on, label, text) => { b.classList.toggle('on', on); b.textContent = label + ': ' + (text || (on ? 'on' : 'off')); };
const isOn = b => b.classList.contains('on');
const speed = document.getElementById('speed');
speed.oninput = () => { const n = Math.round(10 ** speed.value); document.getElementById('sv').textContent = n; ctl('speed=' + n); };
document.getElementById('pause').onclick = e => { paused = !paused; e.target.textContent = paused ? 'Resume' : 'Pause'; ctl('pause=' + (+paused)); };
document.getElementById('step').onclick = () => ctl('step=1');
document.getElementById('stop').onclick = () => ctl('stop=1');
const Lsz = document.getElementById('Lsz'), Nsz = document.getElementById('Nsz'), soupS = document.getElementById('soup');
soupS.oninput = () => { document.getElementById('soupv').textContent = soupS.value + '%'; };
document.getElementById('reset').onclick = () => {                 // a new run, with the size settings
  Lsz.value = Math.max(4, Math.min(512, Math.round(+Lsz.value || 96)));
  Nsz.value = Math.max(0, Math.min(50000000, Math.round(+Nsz.value || 0)));
  ctl(`L=${Lsz.value}&N=${Nsz.value}&soup=${soupS.value / 100}`);
};
const winB = document.getElementById('win'), wrapB = document.getElementById('wrap');
winB.onclick = () => { win = win === 3 ? 2 : 3; setTog(winB, win === 3, 'Window', win === 3 ? '3x3x3' : '2x2x2'); sendRule(); };
wrapB.onclick = () => { setTog(wrapB, !isOn(wrapB), 'Wrap edges'); ctl('wrap=' + (+isOn(wrapB))); };
['b0', 'b1', 's0', 's1'].forEach((id, k) => {
  document.getElementById(id).onchange = e => {
    rules[win][k] = Math.max(0, Math.min(win === 3 ? 26 : 7, Math.round(+e.target.value || 0))); sendRule();
  };
});
document.getElementById('preset').onchange = e => { if (e.target.value === '') return; rules[win] = PRESETS[win][+e.target.value][1].slice(); sendRule(); };
const opS = document.getElementById('op');
opS.oninput = () => { OP = opS.value / 100; document.getElementById('opv').textContent = opS.value + '%'; schedule(); };

cv.addEventListener('pointerdown', e => { drag = [e.clientX, e.clientY]; cv.setPointerCapture(e.pointerId); cv.style.cursor = 'grabbing'; });
cv.addEventListener('pointermove', e => {
  if (!drag) return;
  az += (e.clientX - drag[0]) * 0.01;
  tilt = Math.max(-1.55, Math.min(1.55, tilt + (e.clientY - drag[1]) * 0.01));
  drag = [e.clientX, e.clientY]; schedule();
});
const end = () => { drag = null; cv.style.cursor = 'grab'; };
cv.addEventListener('pointerup', end); cv.addEventListener('pointercancel', end);
cv.addEventListener('wheel', e => { e.preventDefault(); zoom = Math.max(0.3, Math.min(10, zoom * Math.exp(-e.deltaY * 0.0015))); schedule(); }, { passive: false });
cv.addEventListener('dblclick', () => { az = 0.6; tilt = 0.4; zoom = 1; schedule(); });
showRule(); schedule(); poll();
</script></body></html>"""

if 'server' in globals():                        # rerunning the cell: stop the previous sim + server
    state['alive'] = False
    server.shutdown()
    server.server_close()
for th in threading.enumerate():                 # and wait for every earlier sim thread to end, so two never
    if th.name == 'sim_loop' or th.name.endswith('(sim_loop)'):   # share the GPU (one started by an older
        th.join(timeout=30)                      # version of this cell reads the global `state`, so it would
                                                 # otherwise pick up the new one below and run on)

state = dict(alive=True, speed=1, pause=False, step=False, reset=False, error=None, frame=(b'', {}), fid=0,
             want=True, notice='', L=96, N=20_000, soup=0.4, window=Life.window, wrap=Life.wrap,
             rules={k: list(v) for k, v in RULES.items()})

def sim_loop(state):
    """All CUDA work happens on this one thread; the web server only reads state['frame']. A frame is
    packed only once the browser has the previous one (state['want']). It gets its own `state`, so a
    rerun of the cell can't revive it."""
    sim = Life(L=state['L'], N=state['N'], soup=state['soup'])
    dirty = True
    try:
        while state['alive']:
            t_start = time.perf_counter()
            if state['reset']:
                state['reset'] = False
                free = cuda.current_context().get_memory_info()[0]
                if need(state['L']) > free + 0.9 * need(sim.L):          # wouldn't fit even after freeing
                    state['notice'] = (f"a {state['L']}³ box needs {need(state['L']) / 2**20:,.0f} MB of GPU "
                                       f"memory, only ~{(free + need(sim.L)) / 2**20:,.0f} MB free: kept the old box")
                    state['L'] = sim.L
                else:
                    state['notice'] = ''
                    sim = None                                         # free the old grids first, so a big
                    gc.collect()                                       # box never needs room for two
                    cuda.current_context().deallocations.clear()
                    sim = Life(L=state['L'], N=state['N'], soup=state['soup'], seed=int(time.time()) % 10_000)
                dirty = True
            sim.window, sim.wrap = state['window'], state['wrap']
            sim.rules = {k: list(v) for k, v in state['rules'].items()}
            if not state['pause']:
                sim.run(state['speed'], budget=0.5)   # (a big box stops early, so the controls stay live)
                dirty = True
            elif state['step']:
                sim.run(1)
                dirty = True
            state['step'] = False
            if dirty and state['want']:          # fid tells the browser whether there is anything new
                state['want'], dirty = False, False
                state['frame'], state['fid'] = sim.frame(), state['fid'] + 1
            time.sleep(max(0.0, 1 / 30 - (time.perf_counter() - t_start)))   # ~30 updates/s, no idle gap
    except Exception as e:
        state['error'] = f'{type(e).__name__}: {e}'
        state['alive'] = False

class Handler(BaseHTTPRequestHandler):
    def log_message(self, *args):                # keep the notebook output quiet
        pass

    def do_GET(self):
        url = urlparse(self.path)
        extra = {}
        if url.path == '/':
            body, ctype = PAGE.encode(), 'text/html; charset=utf-8'
        elif url.path == '/frame':
            fid, (data, info) = state['fid'], state['frame']
            have = parse_qs(url.query).get('have', [''])[0]
            if have == str(fid):                 # the browser is up to date: pack the next frame
                state['want'] = True
            body, ctype = (b'' if have == str(fid) else data), 'application/octet-stream'   # empty = unchanged
            extra['X-State'] = json.dumps(dict(info, fid=fid, alive=state['alive'], error=state['error'],
                                               speed=state['speed'], soup=state['soup'], window=state['window'],
                                               wrap=state['wrap'], rules=state['rules'], notice=state['notice']))
        elif url.path == '/ctl':
            q = {k: v[0] for k, v in parse_qs(url.query).items()}
            if 'speed' in q:
                state['speed'] = max(1, int(q['speed']))
            if 'pause' in q:
                state['pause'] = q['pause'] == '1'
            if 'step' in q:
                state['step'] = True
            if 'reset' in q:
                state['reset'] = True
            if 'L' in q:
                state['L'] = min(LMAX, max(4, int(q['L'])))
                state['reset'] = True
            if 'N' in q:
                state['N'] = min(NMAX, max(0, int(q['N'])))
                state['reset'] = True
            if 'soup' in q:
                state['soup'] = min(1.0, max(0.01, float(q['soup'])))
                state['reset'] = True
            if 'window' in q:
                state['window'] = 2 if q['window'] == '2' else 3
            if 'rule' in q:
                nmax = 26 if state['window'] == 3 else 7
                v = [min(nmax, max(0, int(s))) for s in q['rule'].split(',')][:4]
                if len(v) == 4:
                    state['rules'][state['window']] = v
            if 'wrap' in q:
                state['wrap'] = q['wrap'] == '1'
            if 'stop' in q:
                state['alive'] = False
            body, ctype = b'ok', 'text/plain'
        else:
            self.send_error(404)
            return
        self.send_response(200)
        self.send_header('Content-Type', ctype)
        self.send_header('Content-Length', str(len(body)))
        self.send_header('Cache-Control', 'no-store')
        for k, v in extra.items():
            self.send_header(k, v)
        self.end_headers()
        self.wfile.write(body)

class Server(ThreadingHTTPServer):
    allow_reuse_address = False                  # on Windows, reuse would let a 2nd kernel silently share the port

PORT = 8766                                      # (the nucleation notebook uses 8765)
try:
    server = Server(('127.0.0.1', PORT), Handler)
except OSError:                                  # port taken (e.g. another kernel): use any free port
    server = Server(('127.0.0.1', 0), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
threading.Thread(target=sim_loop, args=(state,), name='sim_loop', daemon=True).start()
url = f'http://127.0.0.1:{server.server_port}/'
display(HTML(f'Viewer: <a href="{url}" target="_blank">{url}</a> (open in a browser if the frame below stays blank)'))
display(IFrame(url, width=640, height=820))
